# Liver Disease Prediction 

**What our jupiter  notebook contains:**
- Loads the Indian Liver Patient Dataset
- Cleans and explores the data
- Trains 3 Machine Learning models (Logistic Regression, Decision Tree, Random Forest)
- Compares model performance and selects the best one
- Saves the best model for future predictions
- Makes predictions on new patient data

**Libraries used:** Pandas, NumPy, Matplotlib, Seaborn, Scikit-Learn, Imbalanced-Learn

---

## Step 1: Import Libraries

Before we start, we need to import the Python libraries we will use throughout the project.

- **pandas** — for loading and manipulating data in tables (DataFrames)
- **numpy** — for mathematical operations on arrays
- **matplotlib** and **seaborn** — for creating charts and plots
- **sklearn** (scikit-learn) — for machine learning tools (splitting data, building models, evaluating performance)
- **imblearn** — for SMOTE, which helps balance uneven classes in our dataset
- **joblib** — for saving and loading our trained model to/from a file
- **json** — for saving model information in a readable JSON file
- **os** — for working with file paths
- **re** — for cleaning text using regular expressions

In [ ]:
import os
import re
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: data splitting
from sklearn.model_selection import train_test_split

# Scikit-learn: preprocessing tools
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Scikit-learn: machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Scikit-learn: evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Imbalanced-learn: for SMOTE and a special Pipeline that supports it
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Turn off unnecessary warning messages to keep the output clean
warnings.filterwarnings('ignore')

# Use a clean plot style
if 'seaborn-v0_8-whitegrid' in plt.style.available:
    plt.style.use('seaborn-v0_8-whitegrid')

# Show plots directly in the notebook
%matplotlib inline

print("All libraries imported successfully!")

---
## Step 2: Load the Dataset

Our dataset is a CSV file located in the `dataset/` folder.

We use `encoding='latin1'` because the original CSV file contains some special characters
(non-breaking spaces like `\xa0`) in the column names. The `latin1` encoding handles these correctly.

After loading, we check:
- How many rows and columns the dataset has
- What the column names look like
- A preview of the first few rows

In [ ]:
# Load the dataset from the known location
df_raw = pd.read_csv("dataset/liver_disease_data.csv", encoding='latin1')

# Print the shape (rows, columns)
print("Dataset shape:", df_raw.shape)
print("Number of rows:", df_raw.shape[0])
print("Number of columns:", df_raw.shape[1])

# Print all column names and their data types
print("\nColumn Names and Data Types:")
for i, col in enumerate(df_raw.columns):
    print(f"  [{i}] {repr(col)} - dtype: {df_raw[col].dtype}")

# Show the first 5 rows
df_raw.head()

---
## Step 3: Clean Column Names and Map the Target Variable

### Why clean column names?
The raw dataset has messy column names like `'Age of the patient'`, `'Total Protiens'` (with a typo),
and some names contain invisible special characters (`\xa0` — a non-breaking space).

We need to rename them to short, consistent names like `'age'`, `'total_proteins'`, etc.
This makes the code easier to write and less error-prone.

### How we do it:
1. We define a dictionary (`column_mapping`) that maps each messy name to its clean version.
2. For each column, we remove special characters, convert to lowercase, and look it up in the dictionary.

### Target variable mapping:
In the original dataset, the `Result` column uses:
- `1` = Liver Disease Patient
- `2` = Healthy Patient

We remap this to the standard ML convention:
- `1` = Liver Disease (Positive class — what we want to detect)
- `0` = Healthy (Negative class)

In [ ]:
# Dictionary that maps messy column names to clean names
column_mapping = {
    "age of the patient": "age",
    "gender of the patient": "gender",
    "total bilirubin": "total_bilirubin",
    "direct bilirubin": "direct_bilirubin",
    "alkphos alkaline phosphotase": "alkaline_phosphotase",
    "sgpt alamine aminotransferase": "alamine_aminotransferase",
    "sgot aspartate aminotransferase": "aspartate_aminotransferase",
    "total protiens": "total_proteins",
    "total proteins": "total_proteins",
    "alb albumin": "albumin",
    "a/g ratio albumin and globulin ratio": "albumin_globulin_ratio",
    "result": "result"
}

# Make a copy so we don't modify the original raw data
df = df_raw.copy()

# Clean each column name and rename it using the mapping
cleaned_columns = []
for col in df.columns:
    # Step 1: Replace non-breaking spaces with regular spaces
    cleaned = col.replace('\xa0', ' ')
    # Step 2: Remove leading/trailing spaces and convert to lowercase
    cleaned = cleaned.strip().lower()
    # Step 3: Replace multiple spaces with a single space
    cleaned = re.sub(r'\s+', ' ', cleaned)
    # Step 4: Look up the clean name in our mapping dictionary
    final_name = column_mapping.get(cleaned, cleaned)
    cleaned_columns.append(final_name)

# Apply the cleaned column names
df.columns = cleaned_columns

# Clean the gender column: make sure values are 'Male' or 'Female'
# .str.strip() removes extra spaces, .str.title() capitalizes first letter
df['gender'] = df['gender'].astype(str).str.strip().str.title()
df['gender'] = df['gender'].replace({'Nan': np.nan, 'None': np.nan, '': np.nan})

# Map the target variable: 1 stays as 1 (Diseased), 2 becomes 0 (Healthy)
# First, keep only rows where result is 1 or 2 (remove any invalid values)
df = df[df['result'].isin([1, 2])].reset_index(drop=True)
# Then apply the mapping
df['result'] = df['result'].map({1: 1, 2: 0}).astype(int)

print("Cleaned column names:", list(df.columns))
print("\nTarget value counts after mapping:")
print(df['result'].value_counts())
df.head()

---
## Step 4: Check for Missing Values and Duplicates

### Missing values
Some rows might have blank or missing values (shown as `NaN` in pandas).
We need to know which columns have missing data and how much.

- `df.isnull().sum()` counts the number of missing values in each column
- We also calculate the percentage of missing values

### Duplicate rows
Sometimes the same patient record appears more than once.
We remove duplicates to avoid the model accidentally memorizing repeated data,
which could cause **data leakage** (the same data appearing in both training and test sets).

### Target class distribution
We also check how many patients have liver disease vs. how many are healthy.
If one class is much larger than the other, we have a **class imbalance** problem.
We will handle this later using SMOTE.

In [ ]:
# --- Check for missing values ---
print("--- MISSING VALUES ---")
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df) * 100).round(2)

missing_table = pd.DataFrame({
    "Missing Count": missing_values,
    "Percentage (%)": missing_percent
})

# Show only columns that have missing values
missing_with_data = missing_table[missing_table["Missing Count"] > 0]
if len(missing_with_data) > 0:
    print(missing_with_data)
else:
    print("No missing values found!")

# --- Check for duplicate rows ---
print("\n--- DUPLICATE ROWS ---")
duplicate_count = df.duplicated().sum()
duplicate_percent = (duplicate_count / len(df) * 100)
print(f"Duplicate rows: {duplicate_count} ({duplicate_percent:.2f}%)")

# Remove duplicate rows to prevent data leakage
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f"Dataset size after removing duplicates: {len(df_clean)} rows")

# --- Check target class distribution ---
print("\n--- TARGET CLASS DISTRIBUTION ---")
class_counts = df_clean['result'].value_counts()
class_percent = df_clean['result'].value_counts(normalize=True) * 100

for label in class_counts.index:
    if label == 1:
        name = "Liver Disease (1)"
    else:
        name = "Healthy (0)"
    print(f"  {name}: {class_counts[label]} patients ({class_percent[label]:.2f}%)")

---
## Step 5: Exploratory Data Analysis (EDA)

Before building models, we visualize the data to understand patterns.

We will create 3 visualizations:
1. **Target class distribution** — bar chart and donut chart showing how many patients are diseased vs healthy
2. **Correlation heatmap** — shows which features are related to each other and to the target
3. **Feature distributions** — density plots showing how each feature differs between diseased and healthy patients

In [ ]:
# --- 1. Target Class Distribution (Bar Chart + Donut Chart) ---

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

labels = ["Liver Disease (1)", "Healthy (0)"]
disease_count = df_clean['result'].value_counts()[1]
healthy_count = df_clean['result'].value_counts()[0]
counts = [disease_count, healthy_count]
colors = ["#e74c3c", "#2ecc71"]

# Bar chart
bars = ax1.bar(labels, counts, color=colors, width=0.5, edgecolor="black", alpha=0.85)
ax1.set_title("Target Class Counts", fontsize=13, fontweight="bold")
ax1.set_ylabel("Number of Patients", fontsize=11)
ax1.grid(axis="y", linestyle="--", alpha=0.7)

# Add count labels on each bar
for bar in bars:
    height = bar.get_height()
    percent = (height / len(df_clean)) * 100
    ax1.annotate(
        f"{height:,}\n({percent:.1f}%)",
        (bar.get_x() + bar.get_width() / 2, height / 2),
        ha="center", va="center", color="white", fontweight="bold", fontsize=11
    )

# Donut chart
ax2.pie(
    counts, labels=labels, colors=colors, autopct="%1.1f%%",
    startangle=140, wedgeprops=dict(width=0.4, edgecolor="w", linewidth=2)
)
ax2.set_title("Class Proportion Breakdown", fontsize=13, fontweight="bold")

plt.suptitle("Target Class Distribution: Indian Liver Patient Dataset",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. Correlation Heatmap ---
# This shows how strongly each pair of numerical features is related.
# Values close to +1 or -1 mean strong correlation; values near 0 mean weak correlation.

numerical_features = [
    "age", "total_bilirubin", "direct_bilirubin", "alkaline_phosphotase",
    "alamine_aminotransferase", "aspartate_aminotransferase",
    "total_proteins", "albumin", "albumin_globulin_ratio"
]

# Include the target column to see which features correlate with disease
columns_for_correlation = numerical_features + ['result']
correlation_matrix = df_clean[columns_for_correlation].corr()

# Create a triangle mask (to avoid showing duplicate values)
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    correlation_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax
)
ax.set_title("Correlation Matrix of Hepatic Biomarkers & Diagnosis",
             fontsize=13, fontweight="bold", pad=15)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3. Feature Distributions by Diagnosis ---
# These density plots show how each feature's values differ
# between healthy patients (green) and diseased patients (red).
# If the curves look very different, that feature is useful for prediction.

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_features):
    ax = axes[i]
    sns.kdeplot(
        data=df_clean, x=col, hue='result', common_norm=False,
        palette={1: "#e74c3c", 0: "#2ecc71"}, ax=ax, fill=True, alpha=0.3
    )
    ax.set_title(f"Distribution: {col}", fontsize=11, fontweight="bold")
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    # Use log scale for features with very large values (enzymes can be very high)
    if df_clean[col].max() > 100:
        ax.set_xscale("log")
        ax.set_xlabel(f"{col} (log scale)", fontsize=9)

plt.suptitle("Feature Distributions by Diagnosis (0: Healthy, 1: Diseased)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## Step 6: Prepare Features and Target for Machine Learning

Now we need to separate our data into:
- **Features (X)** — the input columns the model will use to make predictions
- **Target (y)** — the output column we want the model to predict (`result`)

### Feature types:
- **Numerical features** (9 columns): age, total_bilirubin, direct_bilirubin, etc.
  These contain numbers like 65, 0.7, 187.
- **Categorical feature** (1 column): gender.
  This contains text values like 'Male' or 'Female'.

We handle these differently during preprocessing:
- Numerical features get **median imputation** (fill missing values with the median) and **scaling** (standardize to similar ranges)
- Categorical features get **mode imputation** (fill missing with the most common value) and **one-hot encoding** (convert text to numbers)

### Train-Test Split:
We split the data into 80% for training and 20% for testing.

- `stratify=y` makes sure both sets have the same ratio of diseased to healthy patients
- `random_state=42` makes the split reproducible (same result every time)

> **Important:** We split the data BEFORE doing any preprocessing.
> This prevents **data leakage** — the test set must remain completely unseen during training.

In [ ]:
# Define the feature columns and their types
feature_names = [
    "age", "gender", "total_bilirubin", "direct_bilirubin",
    "alkaline_phosphotase", "alamine_aminotransferase",
    "aspartate_aminotransferase", "total_proteins", "albumin",
    "albumin_globulin_ratio"
]

numerical_features = [
    "age", "total_bilirubin", "direct_bilirubin", "alkaline_phosphotase",
    "alamine_aminotransferase", "aspartate_aminotransferase",
    "total_proteins", "albumin", "albumin_globulin_ratio"
]

categorical_features = ["gender"]

# Separate features (X) and target (y)
X = df_clean[feature_names].copy()
y = df_clean['result'].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

# Split into training set (80%) and test set (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y     # Keep the same class ratio in both sets
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nDisease rate in training set: {y_train.mean() * 100:.2f}%")
print(f"Disease rate in test set:     {y_test.mean() * 100:.2f}%")

---
## Step 7: Build the Preprocessing Pipeline

### What is a preprocessing pipeline?
A pipeline chains multiple data transformation steps together. This is useful because:
1. It applies all transformations in the correct order automatically
2. It prevents data leakage by fitting only on training data
3. It makes the code cleaner and less error-prone

### What our pipeline does:

**For numerical features (age, bilirubin, etc.):**
1. **SimpleImputer(strategy='median')** — fills missing values with the median (middle value).
   We use median instead of mean because liver enzyme values often have extreme outliers.
2. **StandardScaler()** — scales values so they all have a similar range (mean=0, std=1).
   This helps models like Logistic Regression work better.

**For categorical features (gender):**
1. **SimpleImputer(strategy='most_frequent')** — fills missing gender values with the most common gender.
2. **OneHotEncoder(drop='first')** — converts 'Male'/'Female' into numbers.
   `drop='first'` means we only keep one column (e.g., 1=Male, 0=Female) to avoid redundancy.

### ColumnTransformer
The `ColumnTransformer` lets us apply different pipelines to different columns.
Numerical columns get the numerical pipeline, and gender gets the categorical pipeline.

> **Why not just do this manually?** Using a pipeline ensures that when we save the model,
> the preprocessing is saved with it. When we make predictions later, the same transformations
> are applied automatically. This is essential for correct predictions.

In [ ]:
# Pipeline for numerical features: fill missing values with median, then scale
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline for categorical features: fill missing with most common, then one-hot encode
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
])

# Combine both pipelines using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"  # Drop any columns not listed above
)

# Test the preprocessor on the training data
X_train_transformed = preprocessor.fit_transform(X_train)
print("Transformed training data shape:", X_train_transformed.shape)
print("Preprocessor built and verified successfully!")

---
## Step 8: Train and Evaluate Three Machine Learning Models

We will train 3 different models and compare their performance:

1. **Logistic Regression** — A simple linear model. Good as a baseline.
2. **Decision Tree** — Makes predictions by learning rules from the data (like a flowchart).
3. **Random Forest** — An ensemble of many decision trees that vote together for better accuracy.

### SMOTE (Synthetic Minority Over-sampling Technique)
Our dataset is imbalanced — there are more liver disease patients than healthy ones.
SMOTE creates synthetic (artificial) examples of the minority class in the **training set only**.
This helps the model learn to detect both classes equally well.

> **Critical:** SMOTE is applied ONLY to training data, NEVER to test data.
> That's why we use `imblearn.pipeline.Pipeline` (called `ImbPipeline` here).
> This special pipeline automatically skips SMOTE when calling `.predict()`.

### How each pipeline works:
```
Training:  Raw Data -> Preprocessing -> SMOTE -> Model Training
Testing:   Raw Data -> Preprocessing -> Model Prediction (no SMOTE!)
```

### Evaluation Metrics:
- **Accuracy** — What percentage of all predictions are correct?
- **Precision** — Of patients predicted as diseased, how many actually are?
- **Recall (Sensitivity)** — Of actual diseased patients, how many did we catch?
- **F1-Score** — Harmonic mean of Precision and Recall (balanced metric)
- **ROC-AUC** — How well can the model distinguish between classes? (1.0 = perfect)
- **Specificity** — Of actual healthy patients, how many were correctly identified?

> **In medical screening, Recall is the most important metric** because
> missing a diseased patient (False Negative) is more dangerous than a false alarm (False Positive).

In [ ]:
# Define the three models with their hyperparameters
# (These are the same hyperparameters used in the original project)
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1500, C=1.0, solver="lbfgs", random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, min_samples_split=20, min_samples_leaf=10, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150, max_depth=12, min_samples_split=10,
        min_samples_leaf=5, random_state=42, n_jobs=-1
    )
}

# Dictionaries to store trained pipelines and their metrics
trained_pipelines = {}
metrics_summary = {}

print("=" * 65)
print("TRAINING ALL MODELS (with SMOTE on training data)")
print("=" * 65)

for name, model in models.items():
    print(f"\n---> Training: {name}...")

    # Build a fresh pipeline for each model:
    # Step 1: Preprocess the data (impute + scale + encode)
    # Step 2: Apply SMOTE to balance the training data
    # Step 3: Train the classifier
    #
    # We create a fresh preprocessor for each model because each pipeline
    # stores its own fitted parameters.
    fresh_preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numerical_features),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
            ]), categorical_features)
        ],
        remainder="drop"
    )

    pipeline = ImbPipeline([
        ("preprocessor", fresh_preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", model)
    ])

    # Train the pipeline on training data
    pipeline.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred = pipeline.predict(X_test)

    # Get probability scores (how confident the model is)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    # Calculate all evaluation metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)

    # Calculate confusion matrix values
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()  # Unpack: True Neg, False Pos, False Neg, True Pos

    # Specificity = True Negatives / (True Negatives + False Positives)
    if (tn + fp) > 0:
        specificity = tn / (tn + fp)
    else:
        specificity = 0.0

    # Store all metrics
    metrics_summary[name] = {
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall (Sensitivity)": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(roc_auc, 4),
        "Specificity": round(specificity, 4),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "Confusion_Matrix": cm.tolist()
    }
    trained_pipelines[name] = pipeline

    # Print results for this model
    print(f"  Accuracy:    {acc * 100:.2f}%")
    print(f"  Precision:   {prec * 100:.2f}%")
    print(f"  Recall:      {rec * 100:.2f}% (Sensitivity)")
    print(f"  F1-Score:    {f1:.4f}")
    print(f"  ROC-AUC:     {roc_auc:.4f}")
    print(f"  Specificity: {specificity * 100:.2f}%")

### Model Comparison Table

Let's display all metrics side by side to compare the models easily.

In [ ]:
# Create a comparison table from the metrics dictionary
comparison_table = pd.DataFrame(metrics_summary).T
comparison_table = comparison_table[
    ["Accuracy", "Precision", "Recall (Sensitivity)", "F1-Score", "ROC-AUC", "Specificity"]
]

print("=" * 50)
print("MODEL COMPARISON TABLE (Test Set)")
print("=" * 50)
comparison_table

---
## Step 9: Visualize Model Performance

We create three types of visualizations:

1. **Bar chart** — comparing all metrics across models (higher is better)
2. **Confusion matrices** — showing exactly how many patients each model classified correctly/incorrectly
3. **ROC curves** — showing the trade-off between True Positive Rate and False Positive Rate
   (a curve closer to the top-left corner is better)

In [ ]:
# --- 1. Multi-Metric Comparison Bar Chart ---

fig, ax = plt.subplots(figsize=(11, 5))
comparison_table[
    ["Accuracy", "Precision", "Recall (Sensitivity)", "F1-Score", "ROC-AUC"]
].plot(kind="bar", ax=ax, width=0.8, colormap="viridis")

ax.set_title("Model Performance Comparison on Test Set",
             fontsize=13, fontweight="bold", pad=15)
ax.set_ylabel("Score (0.0 to 1.0)", fontsize=11)
ax.set_xlabel("Model", fontsize=11)
ax.set_ylim(0, 1.1)
ax.tick_params(axis="x", rotation=0, labelsize=10)
ax.legend(loc="lower right", frameon=True)

# Add score labels on top of each bar
for p in ax.patches:
    height = p.get_height()
    if height > 0.05:
        ax.annotate(
            f"{height:.2f}",
            (p.get_x() + p.get_width() / 2., height),
            ha='center', va='bottom', fontsize=8,
            xytext=(0, 2), textcoords='offset points'
        )

plt.tight_layout()
plt.show()

In [ ]:
# --- 2. Confusion Matrices for All Three Models ---
# Each matrix shows:
#   Top-left (TN):     Healthy patients correctly predicted as Healthy
#   Top-right (FP):    Healthy patients incorrectly predicted as Diseased (False Alarm)
#   Bottom-left (FN):  Diseased patients incorrectly predicted as Healthy (Missed!)
#   Bottom-right (TP): Diseased patients correctly predicted as Diseased

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, metrics) in zip(axes, metrics_summary.items()):
    cm = np.array(metrics["Confusion_Matrix"])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
        xticklabels=["Healthy (0)", "Liver Disease (1)"],
        yticklabels=["Healthy (0)", "Liver Disease (1)"]
    )
    recall_val = metrics['Recall (Sensitivity)'] * 100
    spec_val = metrics['Specificity'] * 100
    ax.set_title(
        f"{name}\nRecall: {recall_val:.1f}% | Specificity: {spec_val:.1f}%",
        fontsize=11, fontweight="bold"
    )
    ax.set_xlabel("Predicted Diagnosis", fontsize=10)
    ax.set_ylabel("Actual Diagnosis", fontsize=10)

plt.suptitle("Confusion Matrices (Test Set)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- 3. ROC Curves ---
# The ROC curve plots True Positive Rate (Recall) vs False Positive Rate.
# A model with AUC close to 1.0 is excellent.
# The dashed line represents random guessing (AUC = 0.5).

fig, ax = plt.subplots(figsize=(8, 6))

for name, pipeline in trained_pipelines.items():
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc:.4f})")

# Plot the random chance line
ax.plot([0, 1], [0, 1], color="grey", lw=1.5, linestyle="--",
        label="Random Chance (AUC = 0.500)")

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
ax.set_ylabel("True Positive Rate (Recall / Sensitivity)", fontsize=11)
ax.set_title("ROC Curves", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

---
## Step 10: Select the Best Model and Save It

### How do we pick the best model?
We use a **composite score** that weighs different metrics based on their importance in medical screening:

$$\text{Composite Score} = 0.40 \times \text{Recall} + 0.35 \times \text{F1-Score} + 0.25 \times \text{ROC-AUC}$$

- **Recall gets the highest weight (40%)** because catching diseased patients is the top priority
- **F1-Score (35%)** balances precision and recall
- **ROC-AUC (25%)** measures overall discrimination ability

### Saving the model
We save the entire pipeline (preprocessing + model) using **joblib**.
This means when we load the model later, it can directly accept raw data
and handle all the preprocessing internally.

We also save metadata (model name, features, metrics) to a JSON file for reference.

In [ ]:
# --- Select the best model using composite scoring ---

best_model_name = None
best_composite_score = -1.0

print("COMPOSITE CLINICAL SCORING:")
print("Formula: 0.40 * Recall + 0.35 * F1-Score + 0.25 * ROC-AUC")
print()

for name, m in metrics_summary.items():
    composite = (
        0.40 * m["Recall (Sensitivity)"]
        + 0.35 * m["F1-Score"]
        + 0.25 * m["ROC-AUC"]
    )
    print(f"  {name}: Composite Score = {composite:.4f}")
    print(f"    Recall: {m['Recall (Sensitivity)']:.4f}, "
          f"F1: {m['F1-Score']:.4f}, AUC: {m['ROC-AUC']:.4f}")
    if composite > best_composite_score:
        best_composite_score = composite
        best_model_name = name

print(f"\n>>> Best Model: {best_model_name} "
      f"(Composite Score: {best_composite_score:.4f})")

# --- Save the best model pipeline ---
os.makedirs("models", exist_ok=True)
model_path = os.path.join("models", "best_liver_model.joblib")
best_pipeline = trained_pipelines[best_model_name]
joblib.dump(best_pipeline, model_path)
print(f"\nModel saved to: {model_path}")

# --- Save metadata (model info, features, metrics) to JSON ---
metadata = {
    "best_model_name": best_model_name,
    "features": feature_names,
    "metrics_summary": metrics_summary,
    "best_model_metrics": metrics_summary[best_model_name],
    "class_mapping": {"0": "Healthy", "1": "Liver Disease Detected"}
}

metadata_path = os.path.join("models", "model_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print(f"Metadata saved to: {metadata_path}")

---
## Step 11: Make Predictions on New Patient Data

Now we build functions that can predict whether a new patient has liver disease.

### What the prediction function does:
1. **Validates the input** — checks that all required fields are present,
   gender is valid, numerical values are in plausible ranges,
   and Direct Bilirubin doesn't exceed Total Bilirubin (which is medically impossible)
2. **Creates a DataFrame** from the patient data
3. **Loads the saved model** and makes a prediction
4. **Returns the result** including:
   - Prediction (0 = Healthy, 1 = Liver Disease)
   - Probability of disease
   - Risk level (Low / Moderate / High)
   - A disclaimer that this is not a real medical diagnosis

### Plausible ranges:
We check that values are within medically possible ranges.
For example, age must be between 1 and 120, total bilirubin between 0.1 and 80 mg/dL, etc.
This catches obvious data entry errors.

In [ ]:
# Plausible ranges for each feature (to catch data entry errors)
plausible_ranges = {
    "age": {"min": 1, "max": 120, "label": "Age (years)"},
    "gender": {"allowed": ["Male", "Female"], "label": "Biological Sex"},
    "total_bilirubin": {"min": 0.1, "max": 80.0, "label": "Total Bilirubin (mg/dL)"},
    "direct_bilirubin": {"min": 0.05, "max": 30.0, "label": "Direct Bilirubin (mg/dL)"},
    "alkaline_phosphotase": {"min": 20.0, "max": 3000.0, "label": "Alkaline Phosphatase (IU/L)"},
    "alamine_aminotransferase": {"min": 5.0, "max": 3000.0, "label": "ALT / SGPT (IU/L)"},
    "aspartate_aminotransferase": {"min": 5.0, "max": 5000.0, "label": "AST / SGOT (IU/L)"},
    "total_proteins": {"min": 1.0, "max": 15.0, "label": "Total Proteins (g/dL)"},
    "albumin": {"min": 0.5, "max": 8.0, "label": "Albumin (g/dL)"},
    "albumin_globulin_ratio": {"min": 0.1, "max": 5.0, "label": "A/G Ratio"}
}


def validate_input(patient_data):
    """
    Check if the patient data is valid.
    Returns (True, []) if valid, or (False, [list of error messages]) if invalid.
    """
    errors = []

    # Check that all required fields are present
    for field in feature_names:
        if field not in patient_data or patient_data[field] is None:
            errors.append(f"Missing required field: '{field}'")
    if errors:
        return False, errors

    # Check gender
    gender = str(patient_data['gender']).strip().title()
    if gender not in ["Male", "Female"]:
        errors.append(f"Invalid gender '{patient_data['gender']}'. Must be 'Male' or 'Female'.")

    # Check numerical values are within plausible ranges
    for feat, limits in plausible_ranges.items():
        if feat == "gender":
            continue
        try:
            val = float(patient_data[feat])
            if val < limits["min"] or val > limits["max"]:
                errors.append(
                    f"{limits['label']} ({val}) is outside plausible range "
                    f"[{limits['min']}, {limits['max']}]."
                )
        except (ValueError, TypeError):
            errors.append(f"'{feat}' must be a number.")

    # Medical check: Direct Bilirubin cannot be higher than Total Bilirubin
    try:
        total_bil = float(patient_data.get("total_bilirubin", 0))
        direct_bil = float(patient_data.get("direct_bilirubin", 0))
        if direct_bil > total_bil + 0.1:
            errors.append(
                f"Direct Bilirubin ({direct_bil} mg/dL) cannot exceed "
                f"Total Bilirubin ({total_bil} mg/dL)."
            )
    except Exception:
        pass

    is_valid = len(errors) == 0
    return is_valid, errors


def predict_patient(patient_dict, model_file=model_path):
    """
    Predict whether a patient has liver disease.

    Input: a dictionary with all 10 feature values
    Output: a dictionary with the prediction result
    """
    # Load the saved model
    model = joblib.load(model_file)

    # Validate the input
    is_valid, errors = validate_input(patient_dict)
    if not is_valid:
        return {"success": False, "errors": errors}

    # Create a single-row DataFrame with the correct column order
    row_data = {}
    for f in feature_names:
        row_data[f] = patient_dict[f]
    df_single = pd.DataFrame([row_data])

    # Clean the gender column
    df_single["gender"] = df_single["gender"].astype(str).str.strip().str.title()

    # Convert numerical columns to numbers
    for col in numerical_features:
        df_single[col] = pd.to_numeric(df_single[col], errors="coerce")

    # Make prediction
    prediction = int(model.predict(df_single)[0])
    probability = float(model.predict_proba(df_single)[0][1])

    # Determine risk level based on probability
    if probability < 0.30:
        risk = "Low Risk"
    elif probability < 0.60:
        risk = "Moderate / Borderline Risk"
    else:
        risk = "High Risk"

    # Determine label
    if prediction == 1:
        label = "Liver Disease Detected"
    else:
        label = "Healthy / No Liver Disease Detected"

    result = {
        "success": True,
        "prediction": prediction,
        "label": label,
        "probability": round(probability, 4),
        "risk_level": risk,
        "disclaimer": (
            "[NOTICE]: This is an experimental machine learning estimate "
            "for educational purposes, not a medical diagnosis."
        )
    }
    return result

print("Prediction functions defined successfully!")

---
## Step 12: Test the Prediction System

We test our prediction function with three cases:

1. **Case 1 — Sick patient:** A 62-year-old male with elevated bilirubin and liver enzymes.
   We expect the model to predict Liver Disease.

2. **Case 2 — Healthy patient:** A 28-year-old female with normal biomarker values.
   We expect the model to predict Healthy.

3. **Case 3 — Invalid input:** Missing fields, impossible age (195), and Direct Bilirubin > Total Bilirubin.
   We expect the validation to catch these errors.

In [ ]:
# === CASE 1: Sick Patient (elevated biomarkers) ===
patient_sick = {
    "age": 62,
    "gender": "Male",
    "total_bilirubin": 10.9,
    "direct_bilirubin": 5.5,
    "alkaline_phosphotase": 699,
    "alamine_aminotransferase": 64,
    "aspartate_aminotransferase": 100,
    "total_proteins": 7.5,
    "albumin": 3.2,
    "albumin_globulin_ratio": 0.74
}

result1 = predict_patient(patient_sick)
print("=== CASE 1: SICK PATIENT ===")
print("Predicted Class:", result1["prediction"])
print("Diagnosis:", result1["label"])
print(f"Disease Probability: {result1['probability'] * 100:.2f}%")
print("Risk Level:", result1["risk_level"])
print(result1["disclaimer"])

In [ ]:
# === CASE 2: Healthy Patient (normal biomarkers) ===
patient_healthy = {
    "age": 28,
    "gender": "Female",
    "total_bilirubin": 0.7,
    "direct_bilirubin": 0.1,
    "alkaline_phosphotase": 130,
    "alamine_aminotransferase": 18,
    "aspartate_aminotransferase": 20,
    "total_proteins": 7.1,
    "albumin": 4.2,
    "albumin_globulin_ratio": 1.35
}

result2 = predict_patient(patient_healthy)
print("=== CASE 2: HEALTHY PATIENT ===")
print("Predicted Class:", result2["prediction"])
print("Diagnosis:", result2["label"])
print(f"Disease Probability: {result2['probability'] * 100:.2f}%")
print("Risk Level:", result2["risk_level"])

In [ ]:
# === CASE 3: Invalid Input (should catch errors) ===
patient_invalid = {
    "age": 195,                    # Impossible age
    "gender": "Unknown",           # Invalid gender
    "total_bilirubin": 1.0,
    "direct_bilirubin": 8.0        # Direct > Total (medically impossible)
    # Missing several required fields!
}

result3 = predict_patient(patient_invalid)
print("=== CASE 3: INVALID INPUT ===")
print("Success:", result3["success"])
print("Errors caught:")
for error in result3["errors"]:
    print(f"  - {error}")

---
### Key methodologies:
1. **Median imputation** is better than mean for liver enzyme data because these features have extreme outliers
2. **Stratified splitting** ensures both train and test sets have the same disease ratio
3. **Composite scoring** weighs Recall (40%), F1-Score (35%), and ROC-AUC (25%) for clinical model selection
4. **Direct Bilirubin must be less than or equal to Total Bilirubin** — a medical constraint enforced during input validation